<a href="https://colab.research.google.com/github/Keerthanabs1326/Ethnotech_GenAI/blob/main/rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size):
        # Weight matrices
        self.Wxh = np.random.randn(hidden_size, input_size) * 0.01
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.Why = np.random.randn(output_size, hidden_size) * 0.01

        # Bias terms
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

        self.hidden_size = hidden_size



    # Forward Pass

    def forward(self, inputs):

        h = np.zeros((self.hidden_size, 1))  # initial hidden state
        self.last_inputs = inputs
        self.last_hs = {0: h}

        # Process sequence step-by-step
        for t, x in enumerate(inputs):
            x = x.reshape(-1, 1)

            # Hidden state update
            h = np.tanh(
                np.dot(self.Wxh, x) +
                np.dot(self.Whh, h) +
                self.bh
            )

            self.last_hs[t+1] = h

        # Output layer
        y = np.dot(self.Why, h) + self.by
        return y



    # Backward Pass (BPTT)

    def backward(self, d_y, learning_rate=0.01):

        n = len(self.last_inputs)

        # Gradients initialization
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)

        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)

        # Output gradients
        dWhy += np.dot(d_y, self.last_hs[n].T)
        dby += d_y

        # Hidden gradient
        dh = np.dot(self.Why.T, d_y)

        # Backpropagation Through Time
        for t in reversed(range(n)):
            temp = (1 - self.last_hs[t+1] ** 2) * dh

            dbh += temp
            dWxh += np.dot(temp, self.last_inputs[t].reshape(1, -1))
            dWhh += np.dot(temp, self.last_hs[t].T)

            dh = np.dot(self.Whh.T, temp)

        # Update weights using Gradient Descent
        self.Wxh -= learning_rate * dWxh
        self.Whh -= learning_rate * dWhh
        self.Why -= learning_rate * dWhy
        self.bh  -= learning_rate * dbh
        self.by  -= learning_rate * dby

In [ ]:
# Input sequences
X = [
    np.array([[1], [2], [3]]),
    np.array([[2], [3], [4]]),
    np.array([[3], [4], [5]])
]

# Expected outputs (sum)
Y = [
    np.array([[6]]),
    np.array([[9]]),
    np.array([[12]])
]

In [ ]:
# Step 3: Initialize RNN Model

rnn = SimpleRNN(input_size=1, hidden_size=8, output_size=1)



# Step 4: Train the Model

print("Training Started...\n")

for epoch in range(500):

    total_loss = 0

    for x, y_true in zip(X, Y):

        # Forward Pass
        y_pred = rnn.forward(x)

        # Mean Squared Error Loss
        loss = (y_pred - y_true) ** 2
        total_loss += loss

        # Gradient of loss
        d_y = 2 * (y_pred - y_true)

        # Backward Pass
        rnn.backward(d_y)

    # Print loss every 50 epochs
    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss = {total_loss[0][0]:.4f}")

Training Started...

Epoch 0 | Loss = 251.5181
Epoch 50 | Loss = 2.9067
Epoch 100 | Loss = 2.6618
Epoch 150 | Loss = 2.6985
Epoch 200 | Loss = 2.6103
Epoch 250 | Loss = 1.9979
Epoch 300 | Loss = 0.0001
Epoch 350 | Loss = 0.0000
Epoch 400 | Loss = 0.0000
Epoch 450 | Loss = 0.0000


In [ ]:

print("\nTesting the RNN...\n")

test_seq = np.array([[4], [5], [6]])
prediction = rnn.forward(test_seq)

print("Test Input Sequence: [4, 5, 6]")
print("Predicted Output:", prediction[0][0])
print("Expected Output: 15")



Testing the RNN...

Test Input Sequence: [4, 5, 6]
Predicted Output: 13.222236395962604
Expected Output: 15
